# Bench TTFA — objectif « ElevenLabs » (200-500 ms)

Mesure le **time-to-first-audio** de LFM2.5-Audio servi par vLLM-Omni, avec les
deux leviers poussés sur la branche :

| Levier | Où | Effet attendu |
|---|---|---|
| `initial_codec_chunk_frames: 2` | premier chunk émis dès 2 frames (160 ms d'audio) | TTFA plancher 800 ms → ~250-450 ms |
| `enforce_eager: false` (stage 0) | CUDA graphs sur le décode du backbone hybride | ~25 ms/step → ~10 ms/step, RTF < 1 |

Protocole : **cellules 1-5** (setup, idempotent), puis **cellule 6** = bench de
référence (config complète), **cellules 7-8** = A/B (eager, sans chunk initial).
Chaque bench démarre son propre engine (~2-4 min sur T4) — patience.

⏱️ GPU conseillé : T4 suffit ; L4/A100 = chiffres plus proches de la prod.


In [ ]:
# 1. GPU
!nvidia-smi


In [ ]:
# 2. Repo (clone idempotent + pull)
import os
BRANCH = "claude/blissful-tesla-7i1yky"
REPO = "https://github.com/rcarvalo/finetuning_s2s_toolcalling.git"
if not os.path.isdir("/content/finetuning_s2s_toolcalling"):
    !git clone -b {BRANCH} {REPO} /content/finetuning_s2s_toolcalling
%cd /content/finetuning_s2s_toolcalling
!git pull --ff-only
!git log --oneline -3


In [ ]:
# 3. Dépendances.
#    vllm-omni ne déclare pas vllm (versions appariées major.minor) et le build
#    PyPI de vllm 0.22 est CUDA 13 (libcudart.so.13, absent de Colab) →
#    wheel officiel +cu129 du release GitHub + torch assorti (index cu129).
import subprocess, sys
VLLM_WHL = "https://github.com/vllm-project/vllm/releases/download/v0.22.1/vllm-0.22.1+cu129-cp38-abi3-manylinux_2_28_x86_64.whl"
TORCH_IDX = "https://download.pytorch.org/whl/cu129"
!{sys.executable} -m pip install -q "vllm @ {VLLM_WHL}" --extra-index-url {TORCH_IDX}

# Garde-fou : si une tentative précédente a laissé un torch CUDA 13, on le réaligne.
cuda = subprocess.run([sys.executable, "-c", "import torch; print(torch.version.cuda)"],
                      capture_output=True, text=True).stdout.strip()
print("torch CUDA:", cuda or "?")
if cuda.startswith("13"):
    tv = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__.split('+')[0])"],
                        capture_output=True, text=True).stdout.strip()
    !{sys.executable} -m pip install -q --force-reinstall --no-deps "torch=={tv}+cu129" --index-url {TORCH_IDX}
    print("⚠️ torch réaligné sur cu129 — Exécution → Redémarrer la session, puis reprendre ici.")

!{sys.executable} -m pip install -q "vllm-omni==0.22.0" "liquid-audio>=1.3.0"
!{sys.executable} -m pip install -q -e . --no-deps
import importlib.metadata as md
print("vllm", md.version("vllm"), "| vllm-omni", md.version("vllm-omni"),
      "| liquid-audio", md.version("liquid-audio"))


In [ ]:
# 4. Checkpoint converti au layout vLLM-Omni (idempotent)
import os, sys
CKPT = "/content/lfm25_audio_omni"
if not os.path.isdir(CKPT):
    from huggingface_hub import snapshot_download
    base = snapshot_download("LiquidAI/LFM2.5-Audio-1.5B")
    !{sys.executable} -m vllm_omni_lfm2_audio.convert_checkpoint --checkpoint {base} --output {CKPT}
print("checkpoint:", CKPT)


In [ ]:
# 5. Variantes de config pour l'A/B — informatif : chaque cellule de bench
#    régénère SA config depuis configs/vllm_omni_lfm2_audio.yaml (à jour du
#    dernier git pull), donc pas de copie périmée possible.
!mkdir -p /content/bench_cfg
!cp configs/vllm_omni_lfm2_audio.yaml /content/bench_cfg/ref.yaml
!sed 's/enforce_eager: false/enforce_eager: true/' configs/vllm_omni_lfm2_audio.yaml > /content/bench_cfg/eager.yaml
!sed 's/initial_codec_chunk_frames: 2/initial_codec_chunk_frames: 10/' configs/vllm_omni_lfm2_audio.yaml > /content/bench_cfg/big_first_chunk.yaml
!grep -n 'enforce_eager\|initial_codec\|async_scheduling' /content/bench_cfg/*.yaml


## 6. Bench de référence (CUDA graphs + chunk initial court)

⚠️ Si le démarrage échoue pendant la **capture CUDA graph** du stage 0
(crash/garbage à la 1re génération) : passe aux cellules A/B (eager) et
note l'erreur — c'est l'information qu'on cherche.


In [ ]:
# La config est régénérée ICI depuis configs/ (un git pull suffit,
# pas besoin de ré-exécuter la cellule 5 — copies jamais périmées).
!mkdir -p /content/bench_cfg && cp configs/vllm_omni_lfm2_audio.yaml /content/bench_cfg/ref.yaml
import sys
!{sys.executable} scripts/bench_ttfa.py --checkpoint /content/lfm25_audio_omni \
    --deploy-config /content/bench_cfg/ref.yaml --runs 5 --warmup 2


## 7. A/B n°1 — eager (mesure le gain des CUDA graphs)

Même bench, `enforce_eager: true` partout. Compare `TTFA p50` **et** `RTF` :
si RTF > 1 ici et < 1 en référence, les CUDA graphs sont LE levier du
streaming sans trous.

💡 Redémarre le runtime (Exécution → Redémarrer la session) avant chaque
bench si le GPU garde de la mémoire d'un engine précédent (réexécute alors
les cellules 2-3, le checkpoint et les configs sont conservés).


In [ ]:
# La config est régénérée ICI depuis configs/ (un git pull suffit,
# pas besoin de ré-exécuter la cellule 5 — copies jamais périmées).
!mkdir -p /content/bench_cfg && sed 's/enforce_eager: false/enforce_eager: true/' configs/vllm_omni_lfm2_audio.yaml > /content/bench_cfg/eager.yaml
import sys
!{sys.executable} scripts/bench_ttfa.py --checkpoint /content/lfm25_audio_omni \
    --deploy-config /content/bench_cfg/eager.yaml --runs 5 --warmup 2


## 8. A/B n°2 — sans chunk initial court (l'ancien plancher de 800 ms)

`initial_codec_chunk_frames: 10` = comportement d'avant. Le TTFA doit
remonter de ~400-600 ms par rapport à la référence : c'est la part du levier
« premier chunk court » dans le total.


In [ ]:
# La config est régénérée ICI depuis configs/ (un git pull suffit,
# pas besoin de ré-exécuter la cellule 5 — copies jamais périmées).
!mkdir -p /content/bench_cfg && sed 's/initial_codec_chunk_frames: 2/initial_codec_chunk_frames: 10/' configs/vllm_omni_lfm2_audio.yaml > /content/bench_cfg/big_first_chunk.yaml
import sys
!{sys.executable} scripts/bench_ttfa.py --checkpoint /content/lfm25_audio_omni \
    --deploy-config /content/bench_cfg/big_first_chunk.yaml --runs 5 --warmup 2


## 9. Écouter une réponse streamée (sanity check qualité)

Vérifie que le chunk initial de 2 frames ne crée pas d'artefact audible à la
frontière (le chunk suivant arrive avec 13 frames de contexte gauche).


In [ ]:
# Engine in-process (config de référence) + génération streamée
import time, numpy as np
from pathlib import Path
sys.path.insert(0, "scripts")
from bench_ttfa import _load_engine, _wave

omni = _load_engine(Path("/content/lfm25_audio_omni"), Path("/content/bench_cfg/ref.yaml"))


In [ ]:
from IPython.display import Audio, display
from vllm import SamplingParams
from transformers import AutoTokenizer
from vllm_omni_lfm2_audio.constants import IM_END_TOKEN_ID

tok = AutoTokenizer.from_pretrained("/content/lfm25_audio_omni")
SYS = "You are a helpful assistant. Respond with interleaved text and audio."

def stream_say(user):
    ids = tok(f"<|startoftext|><|im_start|>system\n{SYS}<|im_end|>\n"
              f"<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n",
              add_special_tokens=False).input_ids
    sp = [SamplingParams(temperature=0.0, max_tokens=192, stop_token_ids=[IM_END_TOKEN_ID]),
          SamplingParams(max_tokens=1, detokenize=False)]
    t0, ttfa, chunks, text = time.time(), None, [], ""
    for out in omni.generate({"prompt_token_ids": ids}, sp, py_generator=True):
        ro = out.request_output
        if out.final_output_type == "text" and ro and ro.outputs:
            text = ro.outputs[0].text or text
        elif out.final_output_type == "audio":
            w = _wave(getattr(out, "multimodal_output", None) or getattr(ro, "multimodal_output", None))
            if w is not None and w.size:
                if ttfa is None:
                    ttfa = time.time() - t0
                    print(f"⏱️ TTFA = {ttfa*1000:.0f} ms (chunk de {w.size/24000*1000:.0f} ms)")
                chunks.append(w)
    print(f"🤖 {text.strip()}")
    print(f"{len(chunks)} chunks · total {time.time()-t0:.1f}s")
    if chunks:
        display(Audio(np.concatenate(chunks), rate=24000))

stream_say("Bonjour, qui es-tu ?")


## Grille de lecture

| Résultat | Interprétation | Suite (docs/optimization_audit.md) |
|---|---|---|
| TTFA p50 ≤ 500 ms, RTF < 1 | 🎯 objectif tenu (entrée texte) | câbler l'audio-in vLLM (§1.2) pour le tenir en speech-to-speech |
| TTFA ok mais RTF > 1 | trous après le 1er chunk | CUDA graphs depthformer + détokeniseur fp16 (§1.3-1.4) |
| Capture CUDA graph KO au démarrage | hooks vs graph à investiguer | repasser eager (cellule 7) + coller l'erreur dans la conversation |
| TTFA > 500 ms partout | breakdown nécessaire | relancer avec `LFM2_DEBUG_TIMING=1` et regarder prefill vs steps vs détok |

**Rapporte les 3 lignes `TTFA p50=…`** (référence / eager / gros chunk) : elles
décident du prochain chantier.
